In [ ]:
# Mount GDrive -- need test files in your GDrive to work
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# Print Music XML file
import xml.etree as etree
x = etree.parse("/content/gdrive/MyDrive/musescore_sample1.musicxml")
print(etree.tostring(x, pretty_print=True))

b'<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 4.0 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">\n<score-partwise version="4.0">\n  <work>\n    <work-title>Untitled score</work-title>\n    </work>\n  <identification>\n    <creator type="composer">Composer / arranger</creator>\n    <encoding>\n      <software>MuseScore 4.2.1</software>\n      <encoding-date>2024-03-30</encoding-date>\n      <supports element="accidental" type="yes"/>\n      <supports element="beam" type="yes"/>\n      <supports element="print" attribute="new-page" type="yes" value="yes"/>\n      <supports element="print" attribute="new-system" type="yes" value="yes"/>\n      <supports element="stem" type="yes"/>\n      </encoding>\n    </identification>\n  <defaults>\n    <scaling>\n      <millimeters>6.99911</millimeters>\n      <tenths>40</tenths>\n      </scaling>\n    <page-layout>\n      <page-height>1596.77</page-height>\n      <page-width>1233.87</page-width>\n      <page-margins type=

In [9]:
import difflib
import xml.etree.ElementTree as ET
from google.colab import files

def read_file(file_path):
    with open(file_path, 'r') as file:
        return file.readlines()

def compute_diff(file1_lines, file2_lines):
    # Run diff algorithm and get diffs between files formatted as list of lines
    differ = difflib.Differ()
    diff = list(differ.compare(file1_lines, file2_lines))

    old_line_number = -1 # Lines are stored in lists, which are zero-indexed
    new_line_number = -1

    ''' Line numbers of modified file where changes, deletions, and additions occur
    The line number corresponds to the Music XML element that marks the start of the note/rest'''
    change_lines_in_old = []
    deletion_lines_in_old = []

    change_lines_in_new = []
    deletion_lines_in_new = []
    addition_lines_in_new = []

    for i in range(len(diff)): # Iterate through the lines of our diffs list
      line = diff[i]

      # Compute the line number in the two files based on the diffs list
      if line.startswith('- '):
          old_line_number += 1
      elif line.startswith('+ '):
          new_line_number += 1
      elif line.startswith('? '):
          continue
      else:
          old_line_number += 1
          new_line_number += 1

      if (line.startswith('- ') and (line[1:].strip()[1:5] == "step") and
      (diff[i+2][1:].strip()[1:5]) == "step"): # Note changed!
        change_lines_in_old.append(old_line_number + 7)
        '''
          -Line after which to insert the color <notehead> element (must offset by 7 down)
          -Inserts in line after 7 lines after current old_line_number.
          -Inserted <notehead> element comes after <stem> element
        '''
        change_lines_in_new.append(new_line_number + 1 + 7)
        '''
          The reason it's new_line_number + 1 for new file is because the old line number
          is advanced already (since we're at a line that starts with "-"), but
          the new line counter is at the line before the diff region starts. The
          +1 gets us to the line where the new <step> is.

        '''

      elif line.startswith('- ') and (line[1:].strip()[1:6] == "pitch"): # Note deleted!
        deletion_lines_in_old.append(old_line_number + 8)
      elif line.startswith('+ ') and (line[1:].strip()[1:6] == "pitch"): # Note added!
        addition_lines_in_new.append(new_line_number + 8)
      else:
        pass

    lines_in_old_to_annotate = [change_lines_in_old, deletion_lines_in_old]
    lines_in_new_to_annotate = [change_lines_in_new, deletion_lines_in_new, addition_lines_in_new]

    return (''.join(diff), lines_in_old_to_annotate, lines_in_new_to_annotate)

def annotate_files_for_diffs(old_file_lines, new_file_lines,
                                 lines_in_old_to_annotate, lines_in_new_to_annotate,
                                 old_file_name, new_file_name):
    """
    This function takes in the lines of the old and new music XML file, the line numbers
    in each to annotate (highlight/mark with color), and the original names of the files
    and downloads two music XML files: the old and new file each with color annotations
    based on computed diffs.

    There are three diff types: change, deletion, and addition. For changes, the old note is
    colored red and the new note is colored green. For deletions, the old note is colored red.
    For additions, the new note is colored green.

    Args:
      param1 (list): The list of lines (strings) in the old XML file.
      param2 (list): The list of lines (strings) in the new XML file.
      param3 (list): The list of line numbers in the old XML file to (spread across 3 lists, based on the diff type).
      param4 (list): The list of line numbers in the new XML file to (spread across 3 lists, based on the diff type).
      param5 (str): The original name of the old file.
      param6 (str): The original name of the new file.

    Returns:
      None
    """
    change_lines_in_old, deletion_lines_in_old = lines_in_old_to_annotate
    change_lines_in_new, deletion_lines_in_new, addition_lines_in_new = lines_in_new_to_annotate # Unpack the array containing lines in new!

    '''
      Inserting a new <notehead> annotation line to our XML file, as we will do below, pushes
      all the lines after it down, increasing their index by one. In order to insert
      the next annotation at the corret line, then, we must add the offset by which the previous
      insertion(s) have pushed down the line of insertion specified — this is because
      the specified line numbers are passed in in terms of the line numbers before any insertions are made.
    '''
    new_file_offset, old_file_offset = 0, 0

    #COLOR ANNOTATIONS FOR CHANGES:
    for line_no in change_lines_in_old: # Annotate changed notes in old file
      old_file_lines.insert(line_no + old_file_offset,
                            '   <notehead color="#ff0000">normal</notehead>') # Make changed note appear red
      old_file_offset += 1

    for line_no in change_lines_in_new: # Annotate changed notes in new file
      new_file_lines.insert(line_no + new_file_offset,
                            '   <notehead color="#31c854">normal</notehead>') # Make new note appear green
      new_file_offset += 1

    #COLOR ANNOTATIONS FOR DELETIONS:
    for line_no in deletion_lines_in_old: # Annotate deleted notes in old file
      old_file_lines.insert(line_no + old_file_offset,
                            '   <notehead color="#ff0000">normal</notehead>') # Make deletion appear red
      old_file_offset += 1

    #COLOR ANNOTATIONS FOR ADDITIONS:
    for line_no in addition_lines_in_new: # Annotate deleted notes in new file
      new_file_lines.insert(line_no + new_file_offset,
                            '   <notehead color="#31c854">normal</notehead>') # Make new note appear green
      new_file_offset += 1

    # Build annotated old XML file
    old_file_xml_string = '\n'.join(old_file_lines)
    root = ET.fromstring(old_file_xml_string)
    tree = ET.ElementTree(root)
    with open(old_file_name, 'wb') as f:
        tree.write(f, encoding='utf-8', xml_declaration=True)
    files.download(old_file_name)

    # Build annotated new XML file
    new_file_xml_string = '\n'.join(new_file_lines)
    root = ET.fromstring(new_file_xml_string)
    tree = ET.ElementTree(root)
    with open(new_file_name, 'wb') as f:
        tree.write(f, encoding='utf-8', xml_declaration=True)
    files.download(new_file_name)

def main():
    old_file_path = '/content/gdrive/MyDrive/before_addition.musicxml'
    new_file_path = '/content/gdrive/MyDrive/after_addition.musicxml'

    old_file_lines = read_file(old_file_path)
    new_file_lines = read_file(new_file_path)

    diff, lines_in_old_to_annotate, lines_in_new_to_annotate = compute_diff(old_file_lines, new_file_lines)
    print(diff)

    old_file_name = 'old_result.musicxml'
    new_file_name = 'new_result.musicxml'
    annotate_files_for_diffs(old_file_lines, new_file_lines, lines_in_old_to_annotate, lines_in_new_to_annotate, old_file_name, new_file_name)

if __name__ == "__main__":
    main()

  <?xml version="1.0" encoding="UTF-8"?>
  <!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 4.0 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">
  <score-partwise version="4.0">
    <work>
      <work-title>Untitled score</work-title>
      </work>
    <identification>
      <creator type="composer">Composer / arranger</creator>
      <encoding>
        <software>MuseScore 4.2.1</software>
        <encoding-date>2024-04-17</encoding-date>
        <supports element="accidental" type="yes"/>
        <supports element="beam" type="yes"/>
        <supports element="print" attribute="new-page" type="yes" value="yes"/>
        <supports element="print" attribute="new-system" type="yes" value="yes"/>
        <supports element="stem" type="yes"/>
        </encoding>
      </identification>
    <defaults>
      <scaling>
        <millimeters>6.99911</millimeters>
        <tenths>40</tenths>
        </scaling>
      <page-layout>
        <page-height>1596.77</page-height>
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>